# ENN583 Week 12 - Visual Place Recognition

This week's practical compares several approaches to visual place recognition:
* SAD - a lightweight, classical appearance baseline based on normalised pixel differences
* AlexNet - a generic global descriptor from a network pretrained on ImageNet
* DINOv2 global - a modern self-supervised global representation
* DINOv2 + VLAD - an AnyLoc-inspired aggregation of spatial patch features
* CosPlace - a modern global descriptor trained specifically for visual place recognition

To do these experiments, we will use the [VPR Tutorial GitHub repository](https://github.com/stschubert/VPR_Tutorial) and adapt its demo script.

## Step 1: Clone the Visual Place Recognition Tutorial GitHub repository
Clone the [VPR Tutorial repository](https://github.com/stschubert/VPR_Tutorial).

In [ ]:
from pathlib import Path

# Re-running this setup cell should reuse an existing checkout rather than
# failing because the destination directory already exists.
tutorial_dir = Path('VPR_Tutorial')
if tutorial_dir.is_dir():
    print(f'Using existing VPR Tutorial checkout at {tutorial_dir.resolve()}')
else:
    !git clone https://github.com/stschubert/VPR_Tutorial.git

## Step 2: Import the required libraries

In [ ]:
from VPR_Tutorial.evaluation.metrics import createPR, recallAt100precision, recallAtK
from VPR_Tutorial.datasets.load_dataset import GardensPointDataset
import numpy as np

import ipywidgets as widgets
from IPython.display import clear_output, display

from matplotlib import pyplot as plt

from glob import glob
from PIL import Image

## Step 3: Load and visualise the datasets


In [ ]:
# The loader downloads Gardens Point on first use. Matrices use rows for
# database images and columns for query images throughout this notebook.
dataset = GardensPointDataset('VPR_Tutorial/images/GardensPoint/')
imgs_db, imgs_q, GThard, GTsoft = dataset.load()

The gardens point dataset has 200 images in both the database and query set. There is a 1-to-1 correspondence between the database and query set --  images with the same ID were recorded at same places, i.e., the i-th database image matches the i-th query image. You can see this below, in the ground-truth assignment matrix, where there is a ground-truth match between the database and query set along the diagonal.


In [ ]:
# A true entry marks an exact database-query correspondence. For this
# aligned traversal, exact matches form the main diagonal.
plt.imshow(GThard)
plt.colorbar()
plt.xlabel('Query Image Index')
plt.ylabel('Database Image Index')
plt.title('GT Hard Association for Gardens Point Dataset')
plt.show()

Let's view some of the images in the database and query set to get a sense of the data, as well as the main challenge of this dataset!

In [ ]:
# Sample evenly along the route to compare daytime database images with
# their corresponding nighttime queries.
im_idxs = [0, 50, 100, 150, 199]

fig, ax = plt.subplots(2, 5, figsize = (15, 4))
for i, idx in enumerate(im_idxs):

    ax[0, i].imshow(imgs_db[idx])
    ax[1, i].imshow(imgs_q[idx])
    
    # Pixel coordinates do not add useful information to this comparison.
    ax[0, i].set_xticks([])
    ax[1, i].set_xticks([])
    ax[0, i].set_yticks([])
    ax[1, i].set_yticks([])
    
    ax[0, i].set_title(f'Image {idx}')

ax[0, 0].set_ylabel('Database', rotation = 0, labelpad = 50, fontsize = 15)
ax[1, 0].set_ylabel('Query Set', rotation = 0, labelpad = 50, fontsize = 15)

plt.show()

Can you see what the change to visual conditions is that would make this place recognition task challenging?

## Step 4: Load a feature extractor and compute feature descriptors

The main source of information about image correspondences is the image descriptor. A global descriptor represents each image with one vector, making database retrieval efficient. The representation can be a simple appearance vector or a feature learned by a neural network.

Start with the classical SAD baseline, then compare generic AlexNet features, self-supervised DINOv2 features, DINOv2 patch aggregation and the purpose-trained CosPlace descriptor. What changes as the representation and aggregation become more specialised for place recognition?

### What is the classical SAD baseline?

**SAD (Sum of Absolute Differences)** compares image appearance directly rather than using learned features. Each image is converted to grayscale, downsampled and normalised in small patches before being flattened into a vector. For query descriptor $q$ and database descriptor $d$, the distance is

$$D(q,d)=\frac{1}{N}\sum_{i=1}^{N}|q_i-d_i|.$$

A smaller distance indicates a better match. Since the rest of this notebook treats larger values as better similarities, it uses $S(q,d)=-D(q,d)$. Patch normalisation provides some resistance to illumination change, but SAD still compares corresponding image locations directly. It is therefore sensitive to viewpoint change, cropping, camera motion and moving objects.

SAD is useful here because it needs no training or model weights and gives us an interpretable baseline: the learned descriptors should outperform direct pixel-appearance matching when conditions or viewpoints change.

### DINOv2 global and DINOv2 + VLAD

**DINOv2 global** uses the class token from a small self-supervised Vision Transformer as one descriptor for the whole image. Unlike CosPlace, DINOv2 was not trained specifically for place recognition.

**DINOv2 + VLAD** instead retains DINOv2's spatial patch tokens. It learns 16 visual cluster centres from the database patches, assigns each patch to its nearest centre, and sums the residuals between patches and their assigned centres. The normalised residuals are concatenated into one VLAD descriptor. This preserves more information about the local visual patterns present in an image than a single class token.

This is an educational **AnyLoc-inspired** descriptor, not a reproduction of the full AnyLoc system. It uses the small DINOv2 ViT-S/14 model, last-layer patch tokens and a vocabulary learned from this database. The first DINOv2 run downloads the model through PyTorch Hub; no TensorFlow or specialised VLAD package is required.


In [ ]:
# Use an explicit Dropdown instead of relying on interactive() to infer one.
# The inferred widget can leave .result unset in some Jupyter/ipywidgets versions.
descriptor_dropdown = widgets.Dropdown(
    options=['SAD', 'AlexNet', 'DINOv2 global', 'DINOv2 + VLAD', 'CosPlace'],
    value='SAD',
    description='Descriptor:',
    style={'description_width': 'initial'},
)
display(descriptor_dropdown)

In [ ]:
# Read the selected value directly from the dropdown.
descriptor = descriptor_dropdown.value

print(f'===== Compute {descriptor} descriptors')
feature_extractor = None
if descriptor == 'SAD':
    from vpr_feature_extractors import LightweightSAD
    feature_extractor = LightweightSAD()
elif descriptor == 'AlexNet':
    from VPR_Tutorial.feature_extraction.feature_extractor_holistic import AlexNetConv3Extractor
    feature_extractor = AlexNetConv3Extractor()
elif descriptor in {'DINOv2 global', 'DINOv2 + VLAD'}:
    from vpr_feature_extractors import DINOv2FeatureExtractor
    # Both choices share a backbone; only their aggregation differs.
    aggregation = 'global' if descriptor == 'DINOv2 global' else 'vlad'
    feature_extractor = DINOv2FeatureExtractor(aggregation=aggregation)
elif descriptor == 'CosPlace':
    from VPR_Tutorial.feature_extraction.feature_extractor_cosplace import CosPlaceFeatureExtractor
    feature_extractor = CosPlaceFeatureExtractor()
else:
    raise ValueError(f'Unsupported descriptor: {descriptor}')

# Compute the database first: the VLAD extractor learns its vocabulary on
# this call and reuses the same centres for the query descriptors.
print('===== Compute reference set descriptors')
db_D_holistic = feature_extractor.compute_features(imgs_db)
print('===== Compute query set descriptors')
q_D_holistic = feature_extractor.compute_features(imgs_q)
print('===== Completed')

## Step 5. Examine the descriptor comparison and similarity matrix S
To compare database and query descriptors, we use the cosine similarity (e.g. computed by the inner product of the normalized descriptor vectors). We can use this to get a sense of the correspondences being made by our model.

In the following, the dense similarity matrix S between all descriptor pairs is computed.

In [ ]:
def compute_similarity(db_descriptors, query_descriptors, descriptor_name):
    """Return a database-by-query similarity matrix (larger is better)."""
    if descriptor_name == 'SAD':
        # SAD is a distance, so negate it to obtain a similarity score.
        similarities = np.empty((len(db_descriptors), len(query_descriptors)), dtype=np.float32)
        for query_idx, query_descriptor in enumerate(query_descriptors):
            differences = np.abs(db_descriptors - query_descriptor)
            similarities[:, query_idx] = -np.nanmean(differences, axis=1)
        return similarities

    # For learned global descriptors, cosine similarity is the dot product
    # after L2 normalisation.
    db_norms = np.linalg.norm(db_descriptors, axis=1, keepdims=True)
    query_norms = np.linalg.norm(query_descriptors, axis=1, keepdims=True)
    db_normalised = db_descriptors / np.maximum(db_norms, 1e-12)
    query_normalised = query_descriptors / np.maximum(query_norms, 1e-12)
    return db_normalised @ query_normalised.T

print(f'===== Compute similarities for {descriptor}')
S = compute_similarity(db_D_holistic, q_D_holistic, descriptor)

# S has shape (number of database images, number of query images).
fig = plt.imshow(S)
fig.axes.set_xlabel('Query Image Index')
fig.axes.set_ylabel('Database Image Index')
fig.axes.set_title('Similarity matrix S')
plt.colorbar()
plt.show()

### Interactive query explorer

Move the slider to inspect a query and its five highest-ranked database images. Look for perceptual aliasing: are any incorrect places visually convincing?

In [ ]:
# Disable continuous updates so an expensive redraw occurs only after the
# user releases the slider handle.
query_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(imgs_q) - 1,
    step=1,
    description='Query image:',
    continuous_update=False,
    style={'description_width': 'initial'},
)
query_slider.add_class('vpr-query-slider')
query_slider_style = widgets.HTML(
    """<style>
    .vpr-query-slider .widget-label,
    .vpr-query-slider .widget-readout,
    .vpr-query-slider .slider-container {
        font-size: 16px !important;
    }
    </style>"""
)
query_output = widgets.Output()

def show_top_five_matches(change=None):
    query_idx = query_slider.value
    # Similarity is larger-is-better; reverse the ascending argsort result.
    top_indices = np.argsort(S[:, query_idx])[-5:][::-1]

    with query_output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 6, figsize=(16, 3))
        axes[0].imshow(imgs_q[query_idx])
        axes[0].set_title(f'Query {query_idx}')
        axes[0].set_xlabel('Night')

        for rank, db_idx in enumerate(top_indices, start=1):
            axis = axes[rank]
            axis.imshow(imgs_db[db_idx])
            axis.set_title(f'Rank {rank}: DB {db_idx}')
            axis.set_xlabel(f'score = {S[db_idx, query_idx]:.3f}')
            # GTsoft accepts nearby route frames as valid place matches,
            # which is more realistic than requiring an identical frame ID.
            if GTsoft[db_idx, query_idx]:
                for spine in axis.spines.values():
                    spine.set_edgecolor('limegreen')
                    spine.set_linewidth(3)

        for axis in axes:
            axis.set_xticks([])
            axis.set_yticks([])

        correct_rank = np.flatnonzero(GTsoft[top_indices, query_idx])
        if len(correct_rank):
            outcome = f'first acceptable match at rank {correct_rank[0] + 1}'
            outcome_colour = 'darkgreen'
        else:
            outcome = 'no acceptable match in top 5'
            outcome_colour = 'red'
        fig.suptitle(
            f'{descriptor}: {outcome}',
            color=outcome_colour,
            fontsize=16,
            fontweight='bold',
        )
        plt.tight_layout()
        plt.show()

# Register the callback, then draw query zero before the first interaction.
query_slider.observe(show_top_five_matches, names='value')
display(query_slider_style, query_slider, query_output)
show_top_five_matches()

The similarity matrix is strongly descriptor-dependent. With SAD, some queries may have a weak correct match or several similarly scored candidates. A learned descriptor such as AlexNet, DINOv2 or CosPlace may retrieve the same locations much more reliably, so queries 80 and 175 should not be treated as universal failure cases.

**Your turn:** Use the query explorer above to investigate queries 80 and 175.

- Begin with SAD. For query 80, compare the night image with its expected daytime location. Which appearance changes might make direct pixel comparison unreliable?
- For query 175, does SAD give several nearby database images similar scores? What visual repetition or route overlap could explain this?
- Repeat both queries with AlexNet, DINOv2 global, DINOv2 + VLAD and CosPlace. Do the learned representations recover an acceptable match in the top five, and how does its rank or score change?
- Are these methods genuinely resolving the ambiguity, or are they only moving the correct image slightly higher in an otherwise similar ranking?

These are qualitative observations: use the visible scene content and retrieval scores to support your explanation rather than assuming that a high score must be correct.


Any hypotheses on why these particular queries either have low similarity with all of the database, or high similarity with multiple images from the database?

Note that this is qualitative analysis, where we make informed guesses about what's going on with the model, but these may not be true.

## Step 6. Assess Performance

Now that we've examined our data and the results, we can use performance metrics to summarise performance on the dataset. This is useful for comparing methods.

To evaluate the quality of a similarity matrix S, we can apply a series of decreasing thresholds $\theta$ to match more and more image pairs. Combined with ground-truth information, each threshold leads to a different set of true positives, false positives, true negatives and false negatives, which then provides one point on the precision-recall curve.

In the following, the precision-recall curve and the area under the precision-recall curve is computed and visualized for *multi-match VPR*, i.e. all matches between each query image and the database have to be identified.

To understand more about these metrics, read [this tutorial paper](https://arxiv.org/pdf/2303.03281.pdf) (pg.10-11).

In [ ]:
# Sweep a similarity threshold over all possible image pairs. GTsoft
# prevents nearby acceptable frames from being counted as false positives.
P, R = createPR(S, GThard, GTsoft, matching='multi', n_thresh=100)
plt.figure()
plt.plot(R, P)
plt.xlim(0, 1), plt.ylim(0, 1.01)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Result on GardensPoint day_right--night_right')
plt.grid('on')
plt.draw()



The graph shows the precision-recall curve. A curve closer to the upper right corner would represent better performance. Precision=1 means that no false positives (FP) were extracted. A recall=1 means that all same places were found.

Below, we calculate the area under this curve as a summary metric. The AUPRC performance ranges between 0 and 1 (higher is better).

In [ ]:
# Summarise thresholded loop-closure and ranked-retrieval performance.
AUPRC = np.trapz(P, R)
# High precision matters when a false loop closure could corrupt a map.
R100P = recallAt100precision(S, GThard, GTsoft, matching='multi', n_thresh=100)
# Recall@K asks whether any acceptable place occurs among the top K results.
recalls_at_k = {k: recallAtK(S, GTsoft, K=k) for k in (1, 5, 10)}

print(f'===== AUPRC: {AUPRC:.3f}')
print(f'===== Maximum recall at 100% precision: {R100P:.3f}')
print('===== ' + ', '.join(f'Recall@{k}: {value:.3f}' for k, value in recalls_at_k.items()))

Record the metrics for the selected descriptor before comparing methods.

## Step 7. Compare to another method

**Your turn: run the experiment with SAD, AlexNet, DINOv2 global, DINOv2 + VLAD and CosPlace. Predict their relative performance before computing the results.**

You should be able to scroll back up and re-run cells after selecting the new descriptor.

Compare AUPRC, recall at 100% precision and recall@K. Also inspect the similarity matrices and top-five results: does VLAD aggregation improve on the DINOv2 class token? Does a descriptor trained specifically for VPR remove perceptual aliases, or merely rank the correct match slightly higher?

## Step 8. VPR in the Wild!

Collect some of your own images from outside S block at QUT! You can upload these into the 'MyData' folder in the 'images' folder and test them below. I've put a few placeholders, but you should go take your own. Note: you should probably take images in landscape mode with your phone to match the dataset.

**Your turn: The code below will load in your images, extract their features, but is missing the code to compute the similarity matrix! Add this in yourself, then you will be able to visualise the matches for each image.**

How do the models perform?


In [ ]:
# Load the student's own query images. Sorting makes their display order
# deterministic across operating systems and notebook runs.
my_fns_q = sorted(glob('images/*.jpg'))
my_imgs_q = [np.array(Image.open(fn).convert('RGB')) for fn in my_fns_q]

print('===== Compute query set descriptors')
q_my_holistic = feature_extractor.compute_features(my_imgs_q)

# Create S_my using the compute_similarity function defined above.
# TODO: Implement this solution.
pass



The similarity matrix is not very helpful to look at with these images, as we don't necessarily know the ground-truth matches. We can however visualise the matches below!

Were the matches correct? If not, have a look at the similarity of the matches compared to the similarity matrix calculated with the normal query set -- do you think the matches were confident matches?

In [ ]:
# Select the highest-scoring database image independently for each query.
im_matches = np.argmax(S_my, axis=0)

for orig_idx in range(len(im_matches)):
    
    fig, ax = plt.subplots(1, 2, figsize = (6, 4))
    
    ax[0].imshow(imgs_db[im_matches[orig_idx]])
    ax[1].imshow(my_imgs_q[orig_idx])
    
    # Hide pixel coordinates so attention stays on the retrieved content.
    ax[0].set_xticks([])
    ax[1].set_xticks([])
    ax[0].set_yticks([])
    ax[1].set_yticks([])

    ax[0].set_title('Database', rotation = 0, fontsize = 15)
    ax[1].set_title('Query Image', rotation = 0, fontsize = 15)
    
    similarity = np.max(S_my, axis=0)[orig_idx]
    fig.suptitle(f'Similarity of match: {similarity}', y = 0.85, fontsize = 15)
    plt.show()